# Model Preprocessing

In this notebook, we prepare the dataset for machine learning models.

Steps:
1. Load processed dataset (sepsis_final.csv)
2. Train-test split
3. Feature selection using SelectKBest
4. Handle class imbalance using SMOTE
5. Feature scaling for Logistic Regression
6. Save processed data for model training

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

## Load Dataset
We load the processed dataset created from the time-series preprocessing notebook.


In [2]:
df = pd.read_csv("../data/sepsis_final.csv")
print("Dataset Shape:", df.shape)
print("Class distribution:")
print(df["SepsisLabel"].value_counts())
df.head()

Dataset Shape: (706781, 53)
Class distribution:
SepsisLabel
0    696724
1     10057
Name: count, dtype: int64


,Age,Gender,ICULOS,SepsisLabel,patient_id,HR_mean,HR_std,HR_min,HR_max,O2Sat_mean,...,Platelets_min,Platelets_max,Creatinine_mean,Creatinine_std,Creatinine_min,Creatinine_max,Glucose_mean,Glucose_std,Glucose_min,Glucose_max
0,83.14,0,1,0,A_000001,83.833333,4.32435,78.0,90.0,97.5,...,194.0,196.0,0.9,0.0,0.9,0.9,122.5,0.0,117.0,128.0
1,83.14,0,2,0,A_000001,83.833333,4.32435,78.0,90.0,97.5,...,194.0,196.0,0.9,0.0,0.9,0.9,122.5,0.0,117.0,128.0
2,83.14,0,3,0,A_000001,83.833333,4.32435,78.0,90.0,97.5,...,194.0,196.0,0.9,0.0,0.9,0.9,122.5,0.0,117.0,128.0
3,83.14,0,4,0,A_000001,83.833333,4.32435,78.0,90.0,97.5,...,194.0,196.0,0.9,0.0,0.9,0.9,122.5,0.0,117.0,128.0
4,83.14,0,5,0,A_000001,83.833333,4.32435,78.0,90.0,97.5,...,194.0,196.0,0.9,0.0,0.9,0.9,122.5,0.0,117.0,128.0


In [3]:
X = df.drop(["SepsisLabel", "patient_id"], axis=1)
y = df["SepsisLabel"]

# Save feature names for Flask API
feature_names = list(X.columns)
joblib.dump(feature_names, "../models/feature_names.pkl")
print("Features saved:", len(feature_names))
print(feature_names)

Features saved: 51
['Age', 'Gender', 'ICULOS', 'HR_mean', 'HR_std', 'HR_min', 'HR_max', 'O2Sat_mean', 'O2Sat_std', 'O2Sat_min', 'O2Sat_max', 'Temp_mean', 'Temp_std', 'Temp_min', 'Temp_max', 'SBP_mean', 'SBP_std', 'SBP_min', 'SBP_max', 'MAP_mean', 'MAP_std', 'MAP_min', 'MAP_max', 'DBP_mean', 'DBP_std', 'DBP_min', 'DBP_max', 'Resp_mean', 'Resp_std', 'Resp_min', 'Resp_max', 'Lactate_mean', 'Lactate_std', 'Lactate_min', 'Lactate_max', 'WBC_mean', 'WBC_std', 'WBC_min', 'WBC_max', 'Platelets_mean', 'Platelets_std', 'Platelets_min', 'Platelets_max', 'Creatinine_mean', 'Creatinine_std', 'Creatinine_min', 'Creatinine_max', 'Glucose_mean', 'Glucose_std', 'Glucose_min', 'Glucose_max']


We separate input features (X) and target variable (y).
We also remove patient_id because it is not useful for prediction.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)
print("Train class dist:", np.bincount(y_train))
print("Test class dist:", np.bincount(y_test))

Train Shape: (565424, 51)
Test Shape: (141357, 51)
Train class dist: [557378   8046]
Test class dist: [139346   2011]


In [6]:
smote = SMOTE(sampling_strategy=1.0, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Class distribution after SMOTE:")
print(np.bincount(y_train_resampled))
print("Total training samples:", len(y_train_resampled))

Class distribution after SMOTE:
[557378 557378]
Total training samples: 1114756


In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled  = scaler.transform(X_test)

joblib.dump(scaler, "../models/scaler.pkl")
print("Scaler saved.")

Scaler saved.


In [8]:
joblib.dump(X_train_resampled, "../models/X_train.pkl")
joblib.dump(X_test,           "../models/X_test.pkl")
joblib.dump(X_train_scaled,   "../models/X_train_scaled.pkl")
joblib.dump(X_test_scaled,    "../models/X_test_scaled.pkl")
joblib.dump(y_train_resampled,"../models/y_train.pkl")
joblib.dump(y_test,           "../models/y_test.pkl")

print("All preprocessing files saved successfully.")

All preprocessing files saved successfully.


In [9]:
# Save original imbalanced train set for XGBoost
joblib.dump(X_train, "../models/X_train_orig.pkl")
joblib.dump(y_train, "../models/y_train_orig.pkl")

neg = np.bincount(y_train)[0]
pos = np.bincount(y_train)[1]
spw = round(neg / pos)
joblib.dump(spw, "../models/xgb_scale_pos_weight.pkl")
print(f"Original class ratio — neg:{neg}, pos:{pos}, scale_pos_weight={spw}")

Original class ratio — neg:557378, pos:8046, scale_pos_weight=69
